# Energy Forecasting
## UK (London, Bristol, Leeds)

### 1.Daten einlesen und bearbeiten

In [2]:
# Bibliotheken importieren
import pandas as pd
import openpyxl

#### Sunshine Duration Dataframe anpassen

In [3]:
# Daten einlesen Tabellenblatt "Sunshine Duration"
energy_data_sunshine_duration = pd.read_excel('..\\Data\\Raw\\Energy Forecasting Data.xlsx', sheet_name='Sunshine Duration')
energy_data_sunshine_duration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1525 entries, 0 to 1524
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Unnamed: 0                   1524 non-null   datetime64[ns]
 1   Sunshine duration
(minutes)  1523 non-null   object        
 2   Unnamed: 2                   1523 non-null   object        
 3   Unnamed: 3                   1523 non-null   object        
 4   Unnamed: 4                   29 non-null     object        
 5   Unnamed: 5                   28 non-null     object        
 6   Unnamed: 6                   28 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 83.5+ KB


In [4]:
energy_data_sunshine_duration.rename(columns={'Unnamed: 0':'Date'}, inplace=True)
energy_data_sunshine_duration.head()

,Date,Sunshine duration\n(minutes),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaT,London,Bristol,Leeds,NaN,NaN,NaN
1,2019-01-01,3.869503,213.829788,398.4,NaN,NaN,NaN
2,2019-01-02,354.398937,459.626433,457.966666,NaN,NaN,NaN
3,2019-01-03,358.707094,316.008867,358.601773,NaN,NaN,NaN
4,2019-01-04,327.997512,314.861526,112.468083,NaN,NaN,NaN


In [5]:
energy_data_sunshine_duration["Date"] = pd.to_datetime(energy_data_sunshine_duration["Date"])
energy_data_sunshine_duration.head()

,Date,Sunshine duration\n(minutes),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaT,London,Bristol,Leeds,NaN,NaN,NaN
1,2019-01-01,3.869503,213.829788,398.4,NaN,NaN,NaN
2,2019-01-02,354.398937,459.626433,457.966666,NaN,NaN,NaN
3,2019-01-03,358.707094,316.008867,358.601773,NaN,NaN,NaN
4,2019-01-04,327.997512,314.861526,112.468083,NaN,NaN,NaN


In [6]:
# Schritt 1: Erste Zeile entfernen
energy_data_sunshine_duration = energy_data_sunshine_duration.iloc[1:].reset_index(drop=True)

# Schritt 2: Richtige Spaltennamen setzen
energy_data_sunshine_duration.columns = ["Date", "Sunshine_London_per_min", "Sunshine_Bristol_per_min", "Sunshine_Leeds_per_min", "col5", "col6", "col7"]

# Schritt 3: Unnötige Spalten entfernen
energy_data_sunshine_duration = energy_data_sunshine_duration.drop(columns=["col5", "col6", "col7"])
energy_data_sunshine_duration.head()

# Schritt 4: Tagesdaten für jede Stadt extrahieren
london_bristol_leeds_sunshine_daily_data = energy_data_sunshine_duration[['Date', 'Sunshine_London_per_min', 'Sunshine_Bristol_per_min', 'Sunshine_Leeds_per_min']]

In [7]:
# Datensatz nochmals anzeigen und dann die fehlenden Daten berechnen
london_bristol_leeds_sunshine_daily_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1524 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   object        
 2   Sunshine_Bristol_per_min  1522 non-null   object        
 3   Sunshine_Leeds_per_min    1522 non-null   object        
dtypes: datetime64[ns](1), object(3)
memory usage: 47.8+ KB


In [8]:
# Schritt 5: Sunshine-Daten in float umwandeln
london_bristol_leeds_sunshine_daily_data['Sunshine_London_per_min'] = pd.to_numeric(london_bristol_leeds_sunshine_daily_data['Sunshine_London_per_min'], errors='coerce')
london_bristol_leeds_sunshine_daily_data['Sunshine_Bristol_per_min'] = pd.to_numeric(london_bristol_leeds_sunshine_daily_data['Sunshine_Bristol_per_min'], errors='coerce')
london_bristol_leeds_sunshine_daily_data['Sunshine_Leeds_per_min'] = pd.to_numeric(london_bristol_leeds_sunshine_daily_data['Sunshine_Leeds_per_min'], errors='coerce')

london_bristol_leeds_sunshine_daily_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1524 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   float64       
 2   Sunshine_Bristol_per_min  1522 non-null   float64       
 3   Sunshine_Leeds_per_min    1522 non-null   float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 47.8 KB


In [9]:
# Schritt 6: # Letzten zwei Zeilen entfernen
london_bristol_leeds_sunshine_daily_data = london_bristol_leeds_sunshine_daily_data.iloc[:-2].reset_index(drop=True)
london_bristol_leeds_sunshine_daily_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1522 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   float64       
 2   Sunshine_Bristol_per_min  1522 non-null   float64       
 3   Sunshine_Leeds_per_min    1522 non-null   float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 47.7 KB


#### Demand Dataframe anpassen

In [10]:
# Daten einlesen Tabellenblatt "Demand"
energy_data_demand = pd.read_excel('..\\Data\\Raw\\Energy Forecasting Data.xlsx', sheet_name='Demand')
energy_data_demand.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Unnamed: 0           1522 non-null   datetime64[ns]
 1   Total Demand 
(MWh)  1522 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 23.9 KB


In [11]:
# Head anschauen
energy_data_demand.head()

,Unnamed: 0,Total Demand \n(MWh)
0,2019-01-01,25297.250000
1,2019-01-02,31779.833333
2,2019-01-03,33801.020833
3,2019-01-04,34128.791667
4,2019-01-05,31161.395833


In [12]:
#Schritt 1: Spalten sinnvoll benennen
energy_data_demand.columns = ["Date", "Total_Demand_MWh"]

#Schritt 2: Spalten-Werte prüfen
energy_data_demand["Date"] = pd.to_datetime(energy_data_demand["Date"], errors='coerce')
energy_data_demand["Total_Demand_MWh"] = pd.to_numeric(energy_data_demand["Total_Demand_MWh"], errors='coerce')

In [13]:
energy_data_demand.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              1522 non-null   datetime64[ns]
 1   Total_Demand_MWh  1522 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 23.9 KB


#### Daten zusammenführen: Demand & Sunshine_City

In [14]:
energy_data_tmp = london_bristol_leeds_sunshine_daily_data.merge(energy_data_demand, on='Date', how='inner')
energy_data_tmp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1522 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   float64       
 2   Sunshine_Bristol_per_min  1522 non-null   float64       
 3   Sunshine_Leeds_per_min    1522 non-null   float64       
 4   Total_Demand_MWh          1522 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 59.6 KB


#### Temperature Dataframe miteinbinden

In [26]:
# Daten einlesen Tabellenblatt "Temperature"
energy_data_temperature = pd.read_excel('..\\Data\\Raw\\Energy Forecasting Data.xlsx', sheet_name='Temperature')
energy_data_temperature.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6098 entries, 0 to 6097
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Unnamed: 0        6093 non-null   datetime64[ns]
 1   Temperature (C°)  6090 non-null   object        
 2   Unnamed: 2        6090 non-null   object        
 3   Unnamed: 3        6090 non-null   object        
 4   Unnamed: 4        106 non-null    object        
 5   Unnamed: 5        105 non-null    object        
 6   Unnamed: 6        105 non-null    object        
dtypes: datetime64[ns](1), object(6)
memory usage: 333.6+ KB


In [27]:
# 1. Erste Zeile entfernen
energy_data_temperature = energy_data_temperature.iloc[1:].reset_index(drop=True)

# 2. Spalten sinnvoll benennen
energy_data_temperature.columns = ["Datetime", "Temp_London_C", "Temp_Bristol_C", "Temp_Leeds_C", "col4", "col5", "col6"]

# 3. Unnötige Spalten entfernen
energy_data_temperature = energy_data_temperature.drop(columns=["col4", "col5", "col6"])

# 4. Temperaturdaten in float umwandeln
energy_data_temperature['Temp_London_C'] = pd.to_numeric(energy_data_temperature['Temp_London_C'], errors='coerce')
energy_data_temperature['Temp_Bristol_C'] = pd.to_numeric(energy_data_temperature['Temp_Bristol_C'], errors='coerce')
energy_data_temperature['Temp_Leeds_C'] = pd.to_numeric(energy_data_temperature['Temp_Leeds_C'], errors='coerce')

In [28]:
energy_data_temperature.info()
energy_data_temperature.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6097 entries, 0 to 6096
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Datetime        6093 non-null   datetime64[ns]
 1   Temp_London_C   6089 non-null   float64       
 2   Temp_Bristol_C  6089 non-null   float64       
 3   Temp_Leeds_C    6089 non-null   float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 190.7 KB


,Datetime,Temp_London_C,Temp_Bristol_C,Temp_Leeds_C
0,2019-01-01 06:00:00,5.786411,6.371772,7.311828
1,2019-01-01 12:00:00,4.799745,6.046772,6.441828
2,2019-01-01 18:00:00,9.206412,10.525105,7.543495
3,2019-01-02 00:00:00,4.688078,5.221772,2.925162
4,2019-01-02 06:00:00,1.563078,0.980105,0.868495


In [29]:
# 5. Datumsspalte in datetime umwandeln und nur Tagesdaten extrahieren
energy_data_temperature = energy_data_temperature.rename(columns={"Date": "Datetime"})
energy_data_temperature.set_index("Datetime", inplace=True)

# Tagesdurchschnitt
temp_daily_mean = energy_data_temperature.resample("D").mean()
# Tagesminimum
temp_daily_min = energy_data_temperature.resample("D").min()
# Tagesmaximum
temp_daily_max = energy_data_temperature.resample("D").max()

# 6. Tagesdaten zusammenführen
energy_temp_daily = pd.DataFrame({
    "Date": temp_daily_mean.index,
    "Temp_London_mean": temp_daily_mean["Temp_London_C"],
    "Temp_London_min": temp_daily_min["Temp_London_C"],
    "Temp_London_max": temp_daily_max["Temp_London_C"],

    "Temp_Bristol_mean": temp_daily_mean["Temp_Bristol_C"],
    "Temp_Bristol_min": temp_daily_min["Temp_Bristol_C"],
    "Temp_Bristol_max": temp_daily_max["Temp_Bristol_C"],

    "Temp_Leeds_mean": temp_daily_mean["Temp_Leeds_C"],
    "Temp_Leeds_min": temp_daily_min["Temp_Leeds_C"],
    "Temp_Leeds_max": temp_daily_max["Temp_Leeds_C"],
})

energy_temp_daily = energy_temp_daily.reset_index(drop=True)

energy_temp_daily.info()
energy_temp_daily.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               1524 non-null   datetime64[ns]
 1   Temp_London_mean   1523 non-null   float64       
 2   Temp_London_min    1523 non-null   float64       
 3   Temp_London_max    1523 non-null   float64       
 4   Temp_Bristol_mean  1523 non-null   float64       
 5   Temp_Bristol_min   1523 non-null   float64       
 6   Temp_Bristol_max   1523 non-null   float64       
 7   Temp_Leeds_mean    1523 non-null   float64       
 8   Temp_Leeds_min     1523 non-null   float64       
 9   Temp_Leeds_max     1523 non-null   float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 119.2 KB


,Date,Temp_London_mean,Temp_London_min,Temp_London_max,Temp_Bristol_mean,Temp_Bristol_min,Temp_Bristol_max,Temp_Leeds_mean,Temp_Leeds_min,Temp_Leeds_max
0,2019-01-01,6.597523,4.799745,9.206412,7.647883,6.046772,10.525105,7.099051,6.441828,7.543495
1,2019-01-02,3.032661,0.601411,5.278078,2.975938,0.060105,5.641772,2.196412,0.868495,4.050162
2,2019-01-03,2.149745,1.003078,4.128078,1.598855,-0.016562,3.918438,1.242245,-0.128171,2.781829
3,2019-01-04,1.772245,0.823078,3.353078,0.743438,-1.166562,3.188438,1.107245,-1.051505,4.396828
4,2019-01-05,1.449745,-0.451922,4.433078,0.844272,-1.276562,4.408438,3.546412,1.808495,6.348495


In [30]:
# Letzten 2 Tage entfernen
energy_temp_daily = energy_temp_daily.iloc[:-2].reset_index(drop=True)

energy_temp_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               1522 non-null   datetime64[ns]
 1   Temp_London_mean   1522 non-null   float64       
 2   Temp_London_min    1522 non-null   float64       
 3   Temp_London_max    1522 non-null   float64       
 4   Temp_Bristol_mean  1522 non-null   float64       
 5   Temp_Bristol_min   1522 non-null   float64       
 6   Temp_Bristol_max   1522 non-null   float64       
 7   Temp_Leeds_mean    1522 non-null   float64       
 8   Temp_Leeds_min     1522 non-null   float64       
 9   Temp_Leeds_max     1522 non-null   float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 119.0 KB
